In [ ]:
import torch
import os
from tgcn_model import GCN_muti_att
from configs import Config
from mi_dataset_csv import MiDatasetLSM 

directorio_actual = os.getcwd()
proyecto_raiz = os.path.dirname(directorio_actual)
subset = 'asl100'
config_file = os.path.join(directorio_actual, 'configs', '{}.ini'.format(subset))
configs = Config(config_file)
log_interval = configs.log_interval
num_samples = configs.num_samples
hidden_size = configs.hidden_size
drop_p = configs.drop_p
num_stages = configs.num_stages
n_nodes = 83  
n_dims = 3    
pose_data_root = os.path.join(proyecto_raiz, 'Codigos/Dataset')
train_dataset = MiDatasetLSM(root_dir=os.path.join(pose_data_root, 'train'), 
                                 num_samples=num_samples, 
                                 num_nodes=n_nodes, 
                                 num_dims=n_dims)

model = GCN_muti_att(input_feature=num_samples * n_dims, 
                         hidden_feature=hidden_size,
                         num_class=len(train_dataset.classes_), 
                         p_dropout=drop_p, 
                         num_stage=num_stages,
                         num_nodes=n_nodes).cuda()

ruta_pesos = 'checkpoints/asl100/best_model.pth'
model.load_state_dict(torch.load(ruta_pesos,weights_only=True))

# Esto desactiva capas como Dropout o BatchNorm que no deben actuar igual al inferir
model.eval()

if torch.cuda.is_available():
    model = model.cuda()

print("✅ Modelo cargado correctamente.")

✅ Modelo cargado correctamente.


In [ ]:
import cv2
import numpy as npy
import Extraccion_landmarks

cap = cv2.VideoCapture(0)
buffer_landmarks = []

while cap.isOpened():
    ret, frame = cap.read()
    # A. Extraer landmarks con MediaPipe (tú ya tienes esto en 'Extraccion_landmarks.py')
    landmarks = extraer_landmarks(frame) 
    
    # B. Llenar buffer (mantener solo los últimos 32 frames)
    buffer_landmarks.append(landmarks)
    if len(buffer_landmarks) > 32:
        buffer_landmarks.pop(0)
    
    # C. Inferencia cuando el buffer esté lleno
    if len(buffer_landmarks) == 32:
        input_tensor = preparar_tensor(buffer_landmarks) # [1, 83, 96]
        with torch.no_grad():
            output = model(input_tensor.to(device))
            pred = torch.argmax(output, dim=1)
            letra = clases[pred.item()]
            
        # D. Mostrar en pantalla
        cv2.putText(frame, f'Letra: {letra}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow('LSM Real Time', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()